In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


In [178]:
input_path  = "UK-Sanctions-List.csv"

df = pd.read_csv(input_path,header=1)  #header is at index 1
df.head(5)

/var/folders/mm/n6ct31451898z276zsj_w5lm0000gn/T/ipykernel_30071/3459786965.py:3: DtypeWarning: Columns (48,49,50,51,52,53) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path,header=1)  #header is at index 1


,Last Updated,Unique ID,OFSI Group ID,UN Reference Number,Name 6,Name 1,Name 2,Name 3,Name 4,Name 5,...,IMO number,Current owner/operator (s),Previous owner/operator (s),Current believed flag of ship,Previous flags,Type of ship,Tonnage of ship,Length of ship,Year Built,Hull identification number (HIN)
0,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [179]:
print(df.shape)
print(f'Unique entities (Unique ID): {df["Unique ID"].nunique()}')
exact_dups = df.duplicated().sum()
print(f'Exact duplicate rows         : {exact_dups}')
df.drop_duplicates(inplace=True)
print(df.shape)

(57033, 58)
Unique entities (Unique ID): 6046
Exact duplicate rows         : 650
(56383, 58)


In [ ]:
for i, col in enumerate(df.columns, 1):
    idx = df[col].first_valid_index()
    val = df.loc[idx, col] if idx is not None else "All Values NaN"
    print(f'{i}. {col}: {val}')

1. Last Updated: 16/04/2026
2. Unique ID: AFG0001
3. OFSI Group ID: 12703.0
4. UN Reference Number: TAe.010
5. Name 6: HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE
6. Name 1: MOHAMMAD
7. Name 2: HASSAN
8. Name 3:  MUHAMMAD
9. Name 4: Khan
10. Name 5: MARDAN
11. Name type: Primary Name
12. Alias strength: Good quality a.k.a
13. Title: Haji
14. Name non-latin script: حاجی خيرالله و حاجی ستار صرافی
15. Non-latin script type: Arabic
16. Non-latin script language: Arabic
17. Regime Name: The Afghanistan (Sanctions) (EU Exit) Regulations 2020
18. Designation Type: Entity
19. Designation source: UN
20. Sanctions Imposed: Asset freeze
21. Other Information: Afghan Money Service Provider License Number: 044. Haji Khairullah Haji Sattar Money Exchange was used by Taliban leadership to transfer money to Taliban commanders to fund fighters and operations in Afghanistan as of 2011. Active in Kunar Province. Associated with Abdul Sattar Abdul Manan (TAi.162) and Khairullah Barakzai Khudai Nazar (TAi.1

This are the sanction type which are banned from any banking activity.

Asset freeze (This is the most common and absolute ban on all banking/financial services for the target).

Prohibition on correspondent banking

Prohibition on correspondent banking and clearing

Prohibition on Correspondent banking relationships and processing payments

Prohibition on sterling clearing

### Final Field

The dataset has 58 columns from which we have decided to keep only this fields:

| Column | Source Field(s) | Description | Matching Strategy |
|---|---|---|---|
| `Unique_ID` | `Unique ID` | OFSI's stable identifier for the entity. | Primary Key |
| `Entity_Type` | `Designation Type` | Individual, Entity, or Ship. | Filter / Context |
| `Gender` | `Gender` | Male/Female designation for Individuals. | Hard Match (KYC) |
| `Primary_Name` | `Name 1–6` + `Name type` | The canonical, officially designated name. | Fuzzy Match |
| `Aliases` | `Name 1–6` + `Name type` | Deduplicated list of alternative names/spellings. | Fuzzy Match |
| `DOBs` | `D.O.B` | Cleaned dates of birth. Multiple retained if uncertain. | Soft/Hard Match |
| `Associated_Countries` | `Address Country`, `Nationality`, `Country of birth` | Consolidated geographic footprint. | Soft Match |
| `Passport_Numbers` | `Passport number` | Known passport numbers, deduplicated. | Hard Match |
| `National_IDs` | `National Identifier number` | National identity documents (e.g., SSN equivalents). | Hard Match |
| `IMO_Numbers` | `IMO number` | Maritime vessel numbers (Crucial for Ship entities). | Hard Match |
| `Phone_Numbers` | `Phone number` | Cleansed phone numbers (artefacts removed). | Hard Match |
| `Email_Addresses` | `Email address` | Known email addresses. | Hard Match |
| `Positions_Titles` | `Position` | Known roles or titles (Useful for PEP screening). | Contextual |
| `Sanctions_Regime` | `Regime Name` | Legal sanctions regime (e.g., Russia, Cyber). | Compliance |
| `Sanctions_Imposed` | `Sanctions Imposed` | Sanction type applied (e.g., Asset Freeze). | Compliance |




### Missing Value

### Name fields explained

The dataset uses **7 name fields**, which can be confusing:

| Field | Purpose |
|-------|---------|
| `Name 6` | The last name/surname of the individual OR the full name of the entity or ship|
| `Name 1` | First name (populated for individuals; blank for many entities) |
| `Name 2` | The second name (NOT surname) of the individual|
| `Name 3–5` | Additional parts |
| `Name type` | Whether this row's name is a `Primary Name`, `Alias`, or `Primary Name Variation` |
| `Name non-latin script` |  The non-Latin script version of a name |
| `Non-latin script type` | The type of non-Latin script for the name |
| `Non-latin script language` | The language of the non-Latin script name |
| `Alias strength` | `Good quality` vs `Low quality(empty and blank included)` |


In [180]:
for col in ['Name 6','Name 1','Name 2','Name 3','Name 4','Name 5','Name type',
            'Name non-latin script','Regime Name','Designation Type','Sanctions Imposed',
            'Address Country','Phone number','Email address','D.O.B','Nationality(/ies)',
            'Country of birth','National Identifier number','Passport number','IMO number','Position','Title']:
    pct_filled = df[col].notna().mean() * 100
    print(f'{col}: {pct_filled:.1f}% populated')

print(df.drop_duplicates('Unique ID')['Designation Type'].value_counts())


Name 6: 99.8% populated
Name 1: 42.4% populated
Name 2: 21.5% populated
Name 3: 4.6% populated
Name 4: 0.8% populated
Name 5: 0.2% populated
Name type: 99.9% populated
Name non-latin script: 13.7% populated
Regime Name: 100.0% populated
Designation Type: 100.0% populated
Sanctions Imposed: 100.0% populated
Address Country: 78.6% populated
Phone number: 44.3% populated
Email address: 41.5% populated
D.O.B: 44.0% populated
Nationality(/ies): 38.7% populated
Country of birth: 38.9% populated
National Identifier number: 12.4% populated
Passport number: 18.8% populated
IMO number: 1.5% populated
Position: 26.8% populated
Title: 4.4% populated
Designation Type
Individual    3883
Entity        1531
Ship           632
Name: count, dtype: int64


In [202]:
df[['Unique ID','Name 6','Name 1','Name 2','Name 3','Name 4','Name 5','Name type',
            'Name non-latin script','D.O.B','Regime Name','Designation Type','Sanctions Imposed',
            'Address Country','Phone number','Email address','Nationality(/ies)',
            'Country of birth','National Identifier number','Passport number','IMO number','Position','Title']].head(10)

,Unique ID,Name 6,Name 1,Name 2,Name 3,Name 4,Name 5,Name type,Name non-latin script,D.O.B,...,Address Country,Phone number,Email address,Nationality(/ies),Country of birth,National Identifier number,Passport number,IMO number,Position,Title
0,AFG0001,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,Primary Name,حاجی خيرالله و حاجی ستار صرافی,NaN,...,Iran,-59763,helmand_exchange_msp@yahoo.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AFG0001,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,Primary Name,حاجی خيرالله و حاجی ستار صرافی,NaN,...,Afghanistan,'0202-104748,helmand_exchange_msp@yahoo.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AFG0001,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,Primary Name,حاجی خيرالله و حاجی ستار صرافی,NaN,...,Afghanistan,-103495,helmand_exchange_msp@yahoo.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AFG0001,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,Primary Name,حاجی خيرالله و حاجی ستار صرافی,NaN,...,Afghanistan,'0202-104748,helmand_exchange_msp@yahoo.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AFG0001,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,Primary Name,حاجی خيرالله و حاجی ستار صرافی,NaN,...,Afghanistan,-59763,helmand_exchange_msp@yahoo.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,AFG0001,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,Primary Name,حاجی خيرالله و حاجی ستار صرافی,NaN,...,Afghanistan,'0202-104748,helmand_exchange_msp@yahoo.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,AFG0001,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,Primary Name,حاجی خيرالله و حاجی ستار صرافی,NaN,...,Afghanistan,-103495,helmand_exchange_msp@yahoo.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,AFG0001,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,Primary Name,حاجی خيرالله و حاجی ستار صرافی,NaN,...,Afghanistan,'0202-104748,helmand_exchange_msp@yahoo.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,AFG0001,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,Primary Name,حاجی خيرالله و حاجی ستار صرافی,NaN,...,Afghanistan,-59763,helmand_exchange_msp@yahoo.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,AFG0001,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,Primary Name,حاجی خيرالله و حاجی ستار صرافی,NaN,...,Afghanistan,'0202-104748,helmand_exchange_msp@yahoo.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [203]:
df[['Unique ID','Name 6','Name 1','Name 2','Name 3','Name 4','Name 5','Name type',
            'Name non-latin script','D.O.B','Regime Name','Designation Type','Sanctions Imposed',
            'Address Country','Phone number','Email address','Nationality(/ies)',
            'Country of birth','National Identifier number','Passport number','IMO number','Position','Title']].tail(10)

,Unique ID,Name 6,Name 1,Name 2,Name 3,Name 4,Name 5,Name type,Name non-latin script,D.O.B,...,Address Country,Phone number,Email address,Nationality(/ies),Country of birth,National Identifier number,Passport number,IMO number,Position,Title
57023,YEM0014,Al-Atifi,Nasser,NaN,NaN,NaN,NaN,Primary Name Variation,NaN,dd/mm/1969,...,NaN,NaN,NaN,Yemen,NaN,NaN,NaN,NaN,Houthi Defence Minister,NaN
57024,YEM0015,al-Talibi,Muhammad,Ahmad,NaN,NaN,NaN,Primary Name,محمد أحمد الطالبي,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
57025,YEM0015,al-Talbi,Abu,Jaafar,NaN,NaN,NaN,Primary Name Variation,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
57026,YEM0016,Al-Qadari,Muhammad,Ali,NaN,NaN,NaN,Primary Name,حمد ل ع القادريي,NaN,...,NaN,NaN,NaN,Yemen,NaN,NaN,NaN,NaN,Director of Houthi Naval College,NaN
57027,YEM0016,Al-Qadari,Muhammad,Ali,NaN,NaN,NaN,Primary Name,حمد ل ع القادريي,NaN,...,NaN,NaN,NaN,Yemen,NaN,NaN,NaN,NaN,Houthi Coastal Defence Force Chief,NaN
57028,YEM0016,Al-Qadari,Muhammad,Ali,NaN,NaN,NaN,Primary Name,حمد ل ع القادريي,NaN,...,NaN,NaN,NaN,Yemen,NaN,NaN,NaN,NaN,Major General,NaN
57029,YEM0017,Al-Nabi,Muhammad,Fadl,Abd,NaN,NaN,Primary Name,محمد فضل عبد النبي,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Commander of Houthi Naval Forces,NaN
57030,YEM0017,Abdulnabi,Muhammad,Fadl,NaN,NaN,NaN,Primary Name Variation,محمد فضل عبدالنبي,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Commander of Houthi Naval Forces,NaN
57031,YEM0018,Al-HOUTHI,Ali,Hussein,Badr Al Din,NaN,NaN,Primary Name,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
57032,YEM0018,Al-Huthi,Ali,Husayn,NaN,NaN,NaN,Primary Name Variation,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [229]:
id_column = 'Unique ID'
duplicated_rows = df[df.duplicated(subset=[id_column], keep=False)]
if not duplicated_rows.empty:
    sample_id = "AFG0001"
    subset = df[df[id_column] == sample_id]
    print(f"Comparing {len(subset)} rows for Unique ID: {sample_id}\n")
    subset_transposed = subset.transpose()
    subset_transposed.columns = [f"Row {i+1}" for i in range(len(subset))]
    differences = subset_transposed[subset_transposed.nunique(axis=1) > 1]
    print("--- COLUMNS WITH DIFFERENCES (Permutations/Changes) ---")
    print(differences.to_string())
    print("\n--- COLUMNS THAT ARE IDENTICAL (Proof it's the same base data) ---")
    identical_columns = subset_transposed[subset_transposed.nunique(axis=1) <= 1].index.tolist()
    print(identical_columns)

Comparing 720 rows for Unique ID: AFG0001

--- COLUMNS WITH DIFFERENCES (Permutations/Changes) ---
                                                      Row 1                                       Row 2                                       Row 3                                       Row 4                                            Row 5                                       Row 6                                       Row 7                                       Row 8                                       Row 9                                      Row 10                                      Row 11                                      Row 12                                      Row 13                                      Row 14                                      Row 15                                      Row 16                                      Row 17                                      Row 18                                      Row 19                                      Row 20 

#### The DOB has diffrent format

In [ ]:
print(df["D.O.B"].dropna().unique()[:40])


['dd/mm/1958' 'dd/mm/1950' 'dd/mm/1955' 'dd/mm/1956' 'dd/mm/1957'
 'dd/mm/1945' 'dd/mm/1946' 'dd/mm/1947' 'dd/mm/1948' 'dd/mm/1949'
 '30/01/1972' 'dd/mm/1963' 'dd/mm/1953' 'dd/mm/1960' 'dd/mm/1966'
 'dd/mm/1961' 'dd/mm/1965' '12/11/1967' 'dd/mm/1968' 'dd/mm/1969'
 'dd/mm/1975' 'dd/mm/1973' 'dd/mm/1967' '24/10/1971' 'dd/mm/1964'
 '29/09/1963' 'dd/mm/1959' 'dd/mm/1962' 'dd/mm/1970' '1971' '25/12/1970'
 '01/01/1969' '01/01/1967' 'dd/mm/1942' '01/01/1964' '28/08/1965' '1955'
 '1956' '30/07/1969' '22/02/1957']


In [82]:
import re
def classify_dob(s):
    s = str(s).strip()
    if re.match(r'^\d{2}/\d{2}/\d{4}$', s):        return 'Full date (dd/mm/yyyy)'
    if re.match(r'^dd/mm/\d{4}$', s):               return 'Year only (dd/mm placeholder)'
    if re.match(r'^dd/\d{2}/\d{4}$', s):               return 'Month and Year  (mm/yyyy placeholder)'
    if re.match(r'^\d{4}$', s):                     return 'Year only (bare yyyy)'
    if re.search(r'(?i)between|circa|approx', s):   return 'Approximate (text)'
    return 'Other / partial'

dob_series = df['D.O.B'].dropna()
dob_classes = dob_series.apply(classify_dob).value_counts()
print(dob_classes)

print('Sample DOB values by category:')
for cat in dob_classes.index:
    examples = dob_series[dob_series.apply(classify_dob) == cat].unique()[:3]
    print(f'  {cat}: {list(examples)}')

D.O.B
Full date (dd/mm/yyyy)                   16585
Year only (dd/mm placeholder)             7510
Month and Year  (mm/yyyy placeholder)      526
Year only (bare yyyy)                      186
Other / partial                              2
Name: count, dtype: int64
Sample DOB values by category:
  Full date (dd/mm/yyyy): ['30/01/1972', '12/11/1967', '24/10/1971']
  Year only (dd/mm placeholder): ['dd/mm/1958', 'dd/mm/1950', 'dd/mm/1955']
  Month and Year  (mm/yyyy placeholder): ['dd/09/1958', 'dd/08/1977', 'dd/09/1977']
  Year only (bare yyyy): ['1971', '1955', '1956']
  Other / partial: ["'00/00/1975", '15/08/19yy']


### Name type casing inconsistency

In [39]:
print(df['Name type'].value_counts())


Name type
Alias                     34303
Primary name               7244
Primary Name Variation     6153
Primary Name               6034
Primary name variation     3233
ALias                         2
Name: count, dtype: int64


### A ' in front of the phone number 

In [188]:
print("Number of row with Phone Number ",len(df['Phone number'].dropna()))
phones = df['Phone number'].dropna().unique()
phones = pd.Series(phones)
bad = phones[phones.str.startswith("'")]
neg_bad = phones[phones.str.startswith("-")]
print(f'Unique Phone numbers  {len(phones)}')
print(f'Phone numbers with \' {len(bad)}')
print(f'Sample values: {bad.unique().tolist()[:5]}')
print(f'Phone numbers with "-" {len(neg_bad)}')
print(f'Sample values: {neg_bad.unique().tolist()[:5]}')


Number of row with Phone Number  24975
Unique Phone numbers  953
Phone numbers with ' 57
Sample values: ["'0202-104748", "'0300-8209199", "'042-6812081", "'0271-2167285", "'00375(17)7762032"]
Phone numbers with "-" 12
Sample values: ['-59763', '-103495', '-101823', '-222831', '-287']


In [222]:
total_unique_ids = df['Unique ID'].nunique()
uid_only_dups = df.duplicated(subset=['Unique ID']).sum()
print(f'Total number of distinct Unique IDs: {total_unique_ids:,}')
print(f'Rows with a repeated Unique ID: {uid_only_dups:,} (expected — this is the cross-product design)')

Total number of distinct Unique IDs: 6,046
Rows with a repeated Unique ID: 50,337 (expected — this is the cross-product design)


# Data Transformation

In [ ]:
def build_full_name(row: pd.Series) -> str:
    """
    Combines Name 1 through Name 6

    Name structure (from OFSI documentation):
      Name 1 = First name (populated for individuals; blank for many entities)
      Name 2 = The second name (NOT surname) of the individual
      Name 3–5 = Additional parts
      Name 6 = Last Name/ Surname
    """
    name_cols = ['Name 1', 'Name 2', 'Name 3', 'Name 4', 'Name 5', 'Name 6']
    # Filter out nulls and join with a space
    names = [str(row[col]).strip() for col in name_cols if pd.notnull(row[col]) and str(row[col]).strip() != '']
    return ' '.join(names)

In [61]:
temp = df.drop_duplicates(subset=["Name 1"]).head(5)
temp['combined_name'] = temp.apply(build_full_name,axis=1)
temp

,Last Updated,Unique ID,OFSI Group ID,UN Reference Number,Name 6,Name 1,Name 2,Name 3,Name 4,Name 5,...,Current owner/operator (s),Previous owner/operator (s),Current believed flag of ship,Previous flags,Type of ship,Tonnage of ship,Length of ship,Year Built,Hull identification number (HIN),combined_name
0,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE
1061,14/04/2026,AFG0006,7172.0,TAi.002,AKHUND,MOHAMMAD,HASSAN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MOHAMMAD HASSAN AKHUND
1101,29/04/2026,AFG0007,6909.0,TAi.003,MOHAMMAD JAN,Abdul Kabir,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Abdul Kabir MOHAMMAD JAN
1107,29/04/2026,AFG0007,6909.0,TAi.003,ABDUL,KABIR,MUHAMMAD JAN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KABIR MUHAMMAD JAN ABDUL
1113,29/04/2026,AFG0007,6909.0,TAi.003,JAN,ABDUL,KABIR,MUHAMMAD,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ABDUL KABIR MUHAMMAD JAN


In [65]:
def combine_countries(row):
    """Combines all geographic references"""
    cols = ['Address Country', 'Nationality(/ies)', 'Country of birth']
    countries = [str(row[col]).strip() for col in cols if pd.notnull(row[col]) and str(row[col]).strip() != '']
    return countries
def clean_list(item_list):
    """Helper function to remove nulls, deduplicate, and join lists into a string"""
    clean_items = [str(i).strip() for i in item_list if pd.notnull(i) and str(i).strip() != '']
    return ', '.join(sorted(list(set(clean_items))))

In [66]:
temp = df.drop_duplicates(subset=["Name 1"]).head(5)
temp['combined_name'] = temp.apply(combine_countries,axis=1)
temp['combined_name'] = temp['combined_name'].apply(clean_list)
temp

,Last Updated,Unique ID,OFSI Group ID,UN Reference Number,Name 6,Name 1,Name 2,Name 3,Name 4,Name 5,...,Current owner/operator (s),Previous owner/operator (s),Current believed flag of ship,Previous flags,Type of ship,Tonnage of ship,Length of ship,Year Built,Hull identification number (HIN),combined_name
0,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Iran
1061,14/04/2026,AFG0006,7172.0,TAi.002,AKHUND,MOHAMMAD,HASSAN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Afghanistan
1101,29/04/2026,AFG0007,6909.0,TAi.003,MOHAMMAD JAN,Abdul Kabir,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Afghanistan
1107,29/04/2026,AFG0007,6909.0,TAi.003,ABDUL,KABIR,MUHAMMAD JAN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Afghanistan
1113,29/04/2026,AFG0007,6909.0,TAi.003,JAN,ABDUL,KABIR,MUHAMMAD,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Afghanistan


In [67]:
"""Alias                     34303
Primary name               7244
Primary Name Variation     6153
Primary Name               6034
Primary name variation     3233
ALias                         2
"""


def normalise_name_type(name_type: str) -> str:
    """Standardise case variations in Name type field."""
    if pd.isna(name_type):
        return "Unknown"
    nt = str(name_type).strip()
    mapping = {
        "Primary Name"          : "Primary Name",
        "Primary name"          : "Primary Name",        # inconsistent casing fix
        "Primary Name Variation": "Primary Name Variation",  
        "Primary name variation": "Primary Name Variation",  # inconsistent casing fix
        "Alias"                 : "Alias",
        "ALias"                 : "Alias",               # typo fix
    }
    return mapping.get(nt, nt)



In [227]:
print(df['Name type'].value_counts())
print(df['Name type'].isna().value_counts())
print(df[df['Name type'].isna()]["Unique ID"].unique())
temp = df.drop_duplicates(subset=["Name type"]).head(7)
print(temp['Name type'].value_counts())
temp['Name type'] = temp["Name type"].apply(normalise_name_type)
print(temp['Name type'].value_counts())


Name type
Alias                     33689
Primary name               7244
Primary Name Variation     6151
Primary Name               6000
Primary name variation     3233
ALias                         2
Name: count, dtype: int64
Name type
False    56319
True        64
Name: count, dtype: int64
['AFG0006' 'BEL0124' 'GAC0043' 'GAC0044' 'RUS0078' 'RUS0079' 'RUS0081'
 'RUS0084' 'RUS0085' 'RUS0089' 'RUS1055' 'RUS1068' 'RUS3056' 'RUS3058']
Name type
Primary Name              1
Alias                     1
Primary Name Variation    1
Primary name              1
ALias                     1
Primary name variation    1
Name: count, dtype: int64
Name type
Primary Name              2
Alias                     2
Primary Name Variation    2
Unknown                   1
Name: count, dtype: int64


In [ ]:
def clean_dob_for_kyc(dob_str):
    """
    Cleans messy DOB formats from the UK Sanctions list.
    Retains maximum precision without inventing false dates.
    """
    if pd.isna(dob_str):
        return None
        
    s = str(dob_str).strip()
    
    # Full date
    if re.match(r'^\d{2}/\d{2}/\d{4}$', s):
        return s
        
    # year only placeholder
    match_year_ph = re.match(r'^dd/mm/(\d{4})$', s, re.IGNORECASE)
    if match_year_ph:
        return match_year_ph.group(1)
        
    # Month & Year with placeholders
    match_mo_year = re.match(r'^dd/(\d{2})/(\d{4})$', s, re.IGNORECASE)
    if match_mo_year:
        return f"{match_mo_year.group(1)}/{match_mo_year.group(2)}"
        
    # Bare year
    if re.match(r'^\d{4}$', s):
        return s
    s = s.replace("'", "") # Remove rogue single quotes like "'00/00/1975"
    
    # "00/00/1975" -> Extract "1975"
    match_zeros = re.search(r'00/00/(\d{4})', s)
    if match_zeros:
        return match_zeros.group(1)
        
    # "15/08/19yy" -> Standardize unknown years with "XX"
    s = s.replace("yy", "XX").replace("YY", "XX")
    return s



In [94]:
temp = df.copy()
temp['D.O.B'] = temp['D.O.B'].apply(clean_dob_for_kyc)

def classify_dob(s):
    s = str(s).strip()
    if re.match(r'^\d{2}/\d{2}/\d{4}$', s):        return 'Full date (dd/mm/yyyy)'
    if re.match(r'^dd/mm/\d{4}$', s):               return 'Year only (dd/mm placeholder)'
    if re.match(r'^\d{2}/\d{4}$', s):               return 'Month and Year  (mm/yyyy placeholder)'
    if re.match(r'^\d{4}$', s):                     return 'Year only (bare yyyy)'
    if re.search(r'(?i)between|circa|approx', s):   return 'Approximate (text)'
    return 'Other / partial'

dob_series = temp['D.O.B'].dropna()
dob_classes = dob_series.apply(classify_dob).value_counts()
print(dob_classes)

print('Sample DOB values by category:')
for cat in dob_classes.index:
    examples = dob_series[dob_series.apply(classify_dob) == cat].unique()[:3]
    print(f'  {cat}: {list(examples)}')

D.O.B
Full date (dd/mm/yyyy)                   16585
Year only (bare yyyy)                     7697
Month and Year  (mm/yyyy placeholder)      526
Other / partial                              1
Name: count, dtype: int64
Sample DOB values by category:
  Full date (dd/mm/yyyy): ['30/01/1972', '12/11/1967', '24/10/1971']
  Year only (bare yyyy): ['1958', '1950', '1955']
  Month and Year  (mm/yyyy placeholder): ['09/1958', '08/1977', '09/1977']
  Other / partial: ['15/08/19XX']


In [230]:
def clean_phone(phone_str):
    """Removes apostrophes and drops invalid negative reference codes."""
    if pd.isna(phone_str):
        return None
    p = str(phone_str).replace("'", "").strip()
    # drop internal negative ref codes
    if re.match(r'^-\d+$', p):
        return None
    return p

def aggregate_entity(group):
    # Entity Type
    entity_types = group['Designation Type'].dropna().unique()
    entity_type = entity_types[0] if len(entity_types) > 0 else "Unknown"
    if 'Gender' in group.columns:
        genders = group['Gender'].dropna().unique()
        gender = genders[0] if len(genders) > 0 else ""
    else:
        gender = ""
    # Names (Separating Primary from Aliases)
    primary_names = [name for name in group[group['Clean_Name_Type'] == 'Primary Name']['Full_Name'].tolist() if name.strip() != '']
    primary_name = primary_names[0] if primary_names else group['Full_Name'].iloc[0]
    
    all_names = group['Full_Name'].unique().tolist()
    aliases = [n for n in all_names if n != primary_name]
    
    # Aggregating multi-value fields into lists
    dobs = group['Clean_DOB'].tolist()
    countries = []
    for c_list in group['All_Countries_List']:
        countries.extend(c_list)
        
    passports = group['Passport number'].tolist()
    nat_ids = group['National Identifier number'].tolist()
    
    sanctions = group['Sanctions Imposed'].tolist() if 'Sanctions Imposed' in group.columns else []
    phones = [clean_phone(p) for p in group['Phone number'] if pd.notnull(p)] if 'Phone number' in group.columns else []
    emails = group['Email address'].tolist() if 'Email address' in group.columns else []
    imos = group['IMO number'].tolist() if 'IMO number' in group.columns else []
    positions = group['Position'].tolist() if 'Position' in group.columns else []
    
    regimes = group['Regime Name'].dropna().unique()
    regime = regimes[0] if len(regimes) > 0 else ""
    
    return pd.Series({
        'Unique_ID': group['Unique ID'].iloc[0],
        'Entity_Type': entity_type,
        'Primary_Name': primary_name,
        'Aliases': clean_list(aliases),
        'DOBs': clean_list(dobs),
        'Gender': gender,
        'Associated_Countries': clean_list(countries),
        'Passport_Numbers': clean_list(passports),
        'National_IDs': clean_list(nat_ids),
        'IMO_Numbers': clean_list(imos),
        'Phone_Numbers': clean_list(phones),
        'Email_Addresses': clean_list(emails),
        'Positions_Titles': clean_list(positions),
        'Sanctions_Regime': regime,
        'Sanctions_Imposed': clean_list(sanctions)
    })

In [233]:
temp = df.copy()
temp['Full_Name'] = temp.apply(build_full_name, axis=1)
temp['Clean_Name_Type'] = temp['Name type'].apply(normalise_name_type)
temp['Clean_DOB'] = temp['D.O.B'].apply(clean_dob_for_kyc)
temp['All_Countries_List'] = temp.apply(combine_countries, axis=1)
agg_df = temp.groupby('Unique ID').apply(aggregate_entity).reset_index(drop=True)
print(agg_df.head(10))

  Unique_ID Entity_Type                                Primary_Name  \
0   AFG0001      Entity  HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE   
1   AFG0002      Entity                       ROSHAN MONEY EXCHANGE   
2   AFG0003      Entity                       HAQQANI NETWORK (HQN)   
3   AFG0004      Entity                                  RAHAT LTD.   
4   AFG0005      Entity       HAJI BASIR AND ZARJMIL COMPANY HAWALA   
5   AFG0006  Individual                      MOHAMMAD HASSAN AKHUND   
6   AFG0007  Individual                    Abdul Kabir MOHAMMAD JAN   
7   AFG0008  Individual                   Mohammed Omar GHULAM NABI   
8   AFG0009  Individual                       Muhammad Taher Anwari   
9   AFG0010  Individual                     SAYYED MOHAMMED HAQQANI   

                                             Aliases  \
0  Haji Alim Hawala, Haji Hakim Hawala, Haji Khai...   
1  Haji Ahmad Shah Hawala, Maulawi Ahmed Shah Haw...   
2                                                 

/var/folders/mm/n6ct31451898z276zsj_w5lm0000gn/T/ipykernel_30071/3821489848.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  agg_df = temp.groupby('Unique ID').apply(aggregate_entity).reset_index(drop=True)


# Issue Noticed 

 Case inconsistent in the name which leads to many duplicate and also the final screening script to not work properly so made them lower case better and faster matching 

In [245]:
df[df["Unique ID"] == "AFG0010"][["Name 1","Name 2","Name 3","Name 4","Name 5","Name 6","D.O.B","Regime Name","Designation Type"
,"Sanctions Imposed","National Identifier number","Address Country","Nationality(/ies)","Country of birth"]]

,Name 1,Name 2,Name 3,Name 4,Name 5,Name 6,D.O.B,Regime Name,Designation Type,Sanctions Imposed,National Identifier number,Address Country,Nationality(/ies),Country of birth
1147,SAYYED MOHAMMED,NaN,NaN,NaN,NaN,HAQQANI,dd/mm/1965,The Afghanistan (Sanctions) (EU Exit) Regulati...,Individual,Asset freeze|Travel Ban,NaN,NaN,Afghanistan,Afghanistan
1148,SAYYED MOHAMMED,NaN,NaN,NaN,NaN,HAQQANI,dd/mm/1965,The Afghanistan (Sanctions) (EU Exit) Regulati...,Individual,Asset freeze|Travel Ban,NaN,NaN,Afghanistan,Afghanistan
1149,Sayyed,Mohammad,NaN,NaN,NaN,HAQQANI,dd/mm/1965,The Afghanistan (Sanctions) (EU Exit) Regulati...,Individual,Asset freeze|Travel Ban,NaN,NaN,Afghanistan,Afghanistan
1150,Sayyed,Mohammad,NaN,NaN,NaN,HAQQANI,dd/mm/1965,The Afghanistan (Sanctions) (EU Exit) Regulati...,Individual,Asset freeze|Travel Ban,NaN,NaN,Afghanistan,Afghanistan


In [149]:
len(agg_df["Unique_ID"].unique())

6046

In [150]:
len(df["Unique ID"])

57033

In [235]:
clean = pd.read_csv("sanctions_clean.csv")
print(clean.head(10))

  Unique_ID Entity_Type                                Primary_Name  \
0   AFG0001      Entity  haji khairullah haji sattar money exchange   
1   AFG0002      Entity                       roshan money exchange   
2   AFG0003      Entity                       haqqani network (hqn)   
3   AFG0004      Entity                                  rahat ltd.   
4   AFG0005      Entity       haji basir and zarjmil company hawala   
5   AFG0006  Individual                      mohammad hassan akhund   
6   AFG0007  Individual                    abdul kabir mohammad jan   
7   AFG0008  Individual                   mohammed omar ghulam nabi   
8   AFG0009  Individual                       muhammad taher anwari   
9   AFG0010  Individual                     sayyed mohammed haqqani   

                                             Aliases  \
0  haji alim hawala, haji hakim hawala, haji khai...   
1  haji ahmad shah hawala, maulawi ahmed shah haw...   
2                                                N

In [249]:
print("Size of the Data after cleaning ",clean.shape)

population_report = clean.notna().mean() * 100
print(population_report.map("{:.1f}% populated".format))


import re
def classify_dob(s):
    s = str(s).strip()
    
    # Catch comma-separated lists generated by the aggregation function
    if ',' in s:
        return 'List of dates (multiple)'
        
    # Existing single-date patterns
    if re.match(r'^\d{2}/\d{2}/\d{4}$', s):        
        return 'Full date (dd/mm/yyyy)'
    if re.match(r'^dd/mm/\d{4}$', s):               
        return 'Year only (dd/mm placeholder)'
    if re.match(r'^dd/\d{2}/\d{4}$', s):               
        return 'Month and Year (mm/yyyy placeholder)'
    if re.match(r'^\d{4}$', s):                     
        return 'Year only (bare yyyy)'
    if re.search(r'(?i)between|circa|approx', s):   
        return 'Approximate (text)'
        
    return 'Other / partial'
dob_series = clean['DOBs'].dropna()
dob_classes = dob_series.apply(classify_dob).value_counts()
print(dob_classes)

print('Sample DOB values by category:')
for cat in dob_classes.index:
    examples = dob_series[dob_series.apply(classify_dob) == cat].unique()[:3]
    print(f'  {cat}: {list(examples)}')


phones = clean['Phone_Numbers'].dropna()
bad = phones[phones.str.startswith("'")]
print(f'Phone numbers with \' {len(bad)}')
print(f'Sample values: {bad.unique().tolist()[:5]}')


exact_dups = clean.duplicated().sum()
uid_only_dups = clean.duplicated(subset=['Unique_ID']).sum()

print(f'Exact duplicate rows         : {exact_dups}')
print(f'Rows with a repeated Unique ID: {uid_only_dups:,} (expected — this is the cross-product design)')

Size of the Data after cleaning  (6046, 15)
Unique_ID               100.0% populated
Entity_Type             100.0% populated
Primary_Name            100.0% populated
Aliases                  46.2% populated
DOBs                     54.1% populated
Gender                   48.1% populated
Associated_Countries     78.4% populated
Passport_Numbers          9.4% populated
National_IDs              6.2% populated
IMO_Numbers              10.4% populated
Phone_Numbers            10.1% populated
Email_Addresses           8.4% populated
Positions_Titles         43.7% populated
Sanctions_Regime        100.0% populated
Sanctions_Imposed       100.0% populated
dtype: object
DOBs
Full date (dd/mm/yyyy)      2567
Year only (bare yyyy)        448
List of dates (multiple)     239
Other / partial               18
Name: count, dtype: int64
Sample DOB values by category:
  Full date (dd/mm/yyyy): ['29/03/1965', '06/03/1973', '03/08/1986']
  Year only (bare yyyy): ['1961', '1965', '1969']
  List of date